# Formula E — Video Lab (Challenge 3: the Race Control Observer)

You are building the **Video Verifier**: the piece that reads the trackside CCTV and
answers ONE question about a telemetry-flagged stop — *is the racing line still
blocked, or did it clear?* This notebook is your fast feedback loop. By the end you'll
have a prompt that works, and you'll port it into `starter/video_verifier/verifier.py`.

What you'll do here (about 20 minutes, while the rest of the stack builds):
1. **Explore the telemetry** and find a stopped car yourself — then discover why one
   sense isn't enough (a pit stop looks *identical* in telemetry).
2. **Dissect the mosaics** — 24 CCTV cameras tiled into six 2×2 grids, so one Gemini
   call covers four cameras: **24 cameras → 6 calls.**
3. **Prove the time alignment** — that mp4 offset *N* really is race-second *N*.
   Everything you build rests on this.
4. **Make your first Gemini call** on a `gs://` video *slice* — no download, no ffmpeg.
5. **Iterate to the persistence prompt** — this *is* Task 2, done in seconds-per-loop
   instead of minutes-per-Cloud-Run-build.

> The thesis, kept in view the whole way: **deterministic code decides WHEN to spend a
> model call; the model decides WHAT it's looking at.** The telemetry stop is the
> *when*; your prompt is the *what*.

## 0 · Setup  
Auto-detects your project, derives the mosaics path, and pulls the bundled telemetry. Run it and move on.

In [ ]:
import os, json, gzip, time, subprocess, urllib.request

# --- Project auto-detects (env -> ADC -> gcloud). Override by setting it here. ---
PROJECT_ID = ""
if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "") or ""
if not PROJECT_ID:
    try:
        import google.auth; PROJECT_ID = google.auth.default()[1] or ""
    except Exception: pass
if not PROJECT_ID:
    try:
        PROJECT_ID = subprocess.run(["gcloud","config","get-value","project"],
                                    capture_output=True, text=True).stdout.strip()
    except Exception: pass
assert PROJECT_ID, "Could not auto-detect PROJECT_ID - set it manually above."

# Your project's staged mosaics (same-region reads are free).
MOSAICS_BASE = f"gs://{PROJECT_ID}-fe-mosaics/mosaics"
GEM_MODEL = "gemini-3.5-flash"

# The bundled Berlin R10 telemetry (1 Hz). Pull the one file from the repo.
FRAMES = "frames.jsonl.gz"
RAW = "https://raw.githubusercontent.com/haggman/formula-e-race-control-observer/main/simulator/src/frames.jsonl.gz"
if not os.path.exists(FRAMES):
    urllib.request.urlretrieve(RAW, FRAMES)

# Green flag: race-second 0 = 2024-05-12 13:04:00 UTC (= 15:04:00 Berlin, CEST).
from datetime import datetime, timedelta, timezone
GREEN_UTC = datetime(2024, 5, 12, 13, 4, 0, tzinfo=timezone.utc)
print("project:", PROJECT_ID, "| mosaics:", MOSAICS_BASE, "| model:", GEM_MODEL)

## 1 · Explore the telemetry — find the stop yourself

Load the frames and plot car **#7 (Günther)**'s speed. Don't take the incident on
faith — *see* the moment the car stops.

In [ ]:
import matplotlib.pyplot as plt

# frames -> per-car {race_time_s: car_dict}
series, drivers = {}, {}
with gzip.open(FRAMES, "rt") as f:
    for line in f:
        o = json.loads(line); t = o["race_time_s"]
        for cc in o["cars"]:
            series.setdefault(cc["car_number"], {})[t] = cc
            drivers[cc["car_number"]] = cc["driver_name"]

def speed(n, a, b):
    return [t for t in range(a, b+1) if t in series[n]],            [series[n][t]["speed_kmh"] for t in range(a, b+1) if t in series[n]]

ts, sp = speed(7, 600, 820)
plt.figure(figsize=(10,3))
plt.plot(ts, sp); plt.axhline(5, ls="--", c="grey", lw=1)
plt.title(f"#7 {drivers[7]} — speed (km/h)"); plt.xlabel("race-second"); plt.ylabel("km/h")
plt.axvspan(693, 789, color="red", alpha=0.08); plt.tight_layout(); plt.show()

# Find the stop the way the detector does: speed <= 5 km/h held for >= 6 s.
def find_stop(n, thresh=5.0, hold=6):
    run = []
    for t in sorted(series[n]):
        if series[n][t]["speed_kmh"] <= thresh:
            run.append(t)
            if run[-1]-run[0] >= hold: return run[0]
        else: run = []
    return None
print("first sustained stop for #7 at race-second:", find_stop(7))

### The trap that makes this whole system necessary

Now plot car **#33 (Ticktum)** next to #7. Both go to zero. **Telemetry cannot tell
them apart** — but one is a retirement in a live corner and the other is a routine pit
stop. That ambiguity is the entire reason a second sense (the camera) exists.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3), sharey=True)
for a, (n, lbl) in zip(ax, [(7, "#7 Günther — STOPPED ON TRACK"), (33, "#33 Ticktum — PIT LANE")]):
    t0 = find_stop(n); ts, sp = speed(n, t0-15, t0+40)
    a.plot(ts, sp); a.axhline(5, ls="--", c="grey", lw=1); a.set_title(lbl); a.set_xlabel("race-second")
ax[0].set_ylabel("km/h"); plt.tight_layout(); plt.show()
print("Both hit 0 km/h. Identical in telemetry. The correlator's pit-lane guard tells")
print("them apart by GPS; the camera confirms whether the *track* is actually blocked.")

## 2 · Dissect the mosaics — 24 cameras become 6 Gemini calls

The circuit has 24 CCTV cameras. Asking Gemini about each, per incident, is 24 calls.
Instead the cameras are pre-tiled into **six 2×2 mosaics** — four physically-adjacent
cameras per file, in a fixed layout: **TL, TR, BL, BR**. One call covers four cameras.
**24 → 6: a 4× cost and latency win.** The model tells you which *panel* it saw the
incident in; the code maps that panel back to a real camera id.

In [ ]:
GROUPS = {
    "grp_01_cam01_cam02_cam03_cam04": ["Cam01","Cam02","Cam03","Cam04"],
    "grp_02_cam05_cam06_cam07_cam08": ["Cam05","Cam06","Cam07","Cam08"],
    "grp_03_cam09_cam10_cam11_cam12": ["Cam09","Cam10","Cam11","Cam12"],
    "grp_04_cam13_cam14_cam15_cam16": ["Cam13","Cam14","Cam15","Cam16"],
    "grp_05_cam17_cam18_cam19_cam20": ["Cam17","Cam18","Cam19","Cam20"],
    "grp_06_cam21_cam22_cam23_cam24": ["Cam21","Cam22","Cam23","Cam24"],
}
print(f"{len(GROUPS)} groups x 4 cameras = 24 cameras, but only {len(GROUPS)} Gemini calls per incident.")

Pull one mosaic into the runtime and look at a single frame — see the 2×2 grid, and the clock burned into each panel.

In [ ]:
# Download one mosaic and grab a frame with OpenCV (no ffmpeg needed to VIEW).
try:
    import cv2
except Exception:
    subprocess.run(["pip","install","-q","opencv-python-headless"], check=True); import cv2

GID = next(iter(GROUPS)); LOCAL = f"{GID}.mp4"
if not os.path.exists(LOCAL):
    subprocess.run(["gcloud","storage","cp",f"{MOSAICS_BASE}/{GID}.mp4",LOCAL], check=True)

def frame_at(path, offset_s):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(offset_s*fps)); ok, img = cap.read(); cap.release()
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if ok else None

img = frame_at(LOCAL, 693)
plt.figure(figsize=(8,6)); plt.imshow(img); plt.axis("off")
plt.title(f"{GID} @ offset 693s — panels: TL/TR/BL/BR = {GROUPS[GID]}"); plt.show()

## 3 · Prove the time alignment — everything rests on this

The mosaics are **1 FPS starting at race-second 0**, so **mp4 offset N = race-second N**.
That's what lets you ask for "the footage around the stop" with no clock conversion at
all. Don't trust it — prove it. At offset **693**, the burned-in wall-clock should read
**13:15:33 UTC** (green flag 13:04:00 + 693 s) = **15:15:33 Berlin**.

In [ ]:
want = (GREEN_UTC + timedelta(seconds=693))
print(f"offset 693s should show   {want:%H:%M:%S} UTC  =  {want + timedelta(hours=2):%H:%M:%S} Berlin")
img = frame_at(LOCAL, 693)
plt.figure(figsize=(8,6)); plt.imshow(img); plt.axis("off")
plt.title("Read the clock burned into a panel — does it match?"); plt.show()
# (Optional: later in this notebook you can ask Gemini to read the clock and check it
#  programmatically. Seeing it with your own eyes first is the point.)

## 4 · Your first Gemini call — hand it a `gs://` slice

The novel move: point Gemini **straight at the mosaic in the bucket** and pass
`videoMetadata` start/end offsets, so it decodes **only** the window. No download, no
ffmpeg, no frame extraction. Start open-ended — just ask what it sees around the stop.

In [ ]:
try:
    from google import genai
    from google.genai import types
except Exception:
    subprocess.run(["pip","install","-q","google-genai"], check=True)
    from google import genai
    from google.genai import types
import google.auth
GEM = genai.Client(vertexai=True, project=(PROJECT_ID or google.auth.default()[1]), location="global")

def gemini_slice(group_id, t, lead=10, tail=50, prompt="What do you see in this CCTV clip? Is any car stopped on the track by the end?"):
    start, end = max(0, t-lead), t+tail
    vpart = types.Part(file_data=types.FileData(file_uri=f"{MOSAICS_BASE}/{group_id}.mp4", mime_type="video/mp4"),
                       video_metadata=types.VideoMetadata(start_offset=f"{start}s", end_offset=f"{end}s"))
    resp = GEM.models.generate_content(model=GEM_MODEL,
        contents=[types.Content(role="user", parts=[vpart, types.Part(text=prompt)])],
        config=types.GenerateContentConfig(temperature=0.2))
    return resp.text

# Group 02 holds Cam05, where Günther's stop is visible. Ask it, open-ended.
print(gemini_slice("grp_02_cam05_cam06_cam07_cam08", 693))

## 5 · Iterate to the persistence prompt — this IS Task 2

Open-ended is a start, but Race Control needs one clean, structured answer. The question
must judge the **track's state at the END of the window**, not the presence of a car at
a moment — because a car can be stopped at second 5 and gone by second 50. Get *that*
framing right and one answer separates a retirement from a spin-and-recover.

Below is a working structured sweep over all six groups. **This is your Task 2 workbench**
— edit the prompt, rerun, watch the verdicts. When it reliably returns `blocked` for a
real retirement and `cleared` for a recovery, you've got the prompt to port.

In [ ]:
def _parse(text):
    s = (text or "").strip(); a, b = s.find("{"), s.rfind("}")
    if a != -1 and b > a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    return {}

def verify_sweep(t, lead=10, tail=50):
    start, end = max(0, t-lead), t+tail
    print(f"\n### @ t={t}s  (gs:// slice {start}s-{end}s, all {len(GROUPS)} groups)")
    blocked = []
    for gid, cams in GROUPS.items():
        prompt = (
            "You are a race-control video verifier deciding whether a SAFETY CAR is warranted.\n"
            f"Telemetry flagged a car possibly stopped near here around race time ~{t}s.\n"
            f"This is a ~{end-start}s CCTV clip - a 2x2 mosaic: TL={cams[0]}, TR={cams[1]}, BL={cams[2]}, BR={cams[3]}.\n"
            "Judge the TRACK STATE by the END (the call is about the track, not which car):\n"
            "- Car STILL stopped/stranded on/beside the line at the end: blockage=true, cleared=false.\n"
            "- Car appeared but DROVE AWAY / recovered / line clear by the end: blockage=false, cleared=true.\n"
            "- No stopped car at any point: blockage=false, cleared=false.\n"
            'Reply JSON: {"blockage": bool, "cleared": bool, "panel": "TL|TR|BL|BR|none", '
            '"feed_live": bool, "seen_car": <car number if clearly readable else null>, '
            '"what_you_see": str, "confidence": number}')
        vpart = types.Part(file_data=types.FileData(file_uri=f"{MOSAICS_BASE}/{gid}.mp4", mime_type="video/mp4"),
                           video_metadata=types.VideoMetadata(start_offset=f"{start}s", end_offset=f"{end}s"))
        try:
            resp = GEM.models.generate_content(model=GEM_MODEL,
                contents=[types.Content(role="user", parts=[vpart, types.Part(text=prompt)])],
                config=types.GenerateContentConfig(temperature=0.2, response_mime_type="application/json"))
            d = _parse(resp.text)
        except Exception as e:
            print(f"  {gid:34} ERROR: {e}"); continue
        state = "BLOCKED" if d.get("blockage") else ("cleared" if d.get("cleared") else "-")
        print(f"  {gid:34} {state:8} panel={str(d.get('panel','?')):5} seen_car={d.get('seen_car')} conf={d.get('confidence','?')}")
        if d.get("blockage"): blocked.append(gid)
    print("  VERDICT:", ("BLOCKED (" + ", ".join(blocked) + ")") if blocked else "no blockage")
    return blocked

In [ ]:
# The three cases your verifier must get right (expect: blocked / cleared / blocked).
verify_sweep(693)    # Günther #7  -> BLOCKED (retirement, stays put)
verify_sweep(1373)   # Evans   #9  -> cleared (big moment, but drives away)
verify_sweep(1780)   # Mortara #48 -> BLOCKED

## 6 · Port it — take your prompt to `verifier.py`

You've got a prompt that works. Now move it into the real component so the correlator
can call it. In `starter/video_verifier/verifier.py`, fill the three stubs:

- **`_prompt(...)`** — the persistence question + the JSON contract you just tuned.
- **`VideoVerifier._verify_group(...)`** — one Gemini call over one `gs://` slice
  (the video Part + `videoMetadata` offsets above), returning the parsed dict.
- **`VideoVerifier._aggregate(...)`** — fuse the six replies into one `VideoVerdict`
  (`blocked` / `cleared` / `unseen` / `error` — keep "ran and saw nothing" distinct
  from "never ran").

Then test it standalone, no full stack required:

```bash
python -m starter.video_verifier.verifier --at 693 --cars 7        # -> blocked  (Günther)
python -m starter.video_verifier.verifier --at 1698 --cars 17 23   # -> blocked  (Fenestraz + Nato)
python -m starter.video_verifier.verifier --at 1780 --cars 48      # -> blocked  (Mortara)
```

When those pass, restart your correlator (`python -m correlator.service`), jump to the
Günther stop in the console, and watch the recommendation go from **DOUBLE YELLOW** to
**SAFETY CAR · corroborated**. That board lighting up is *your* code authorising the flag.